# Primera competencia

[19:52, 14/9/2026] Caro Franco: Learning rate: cuanto de cada arbol se usa. NO nos sirve tanto penalizacion pero lo tiene. learning rates son constantes. ver animacion y documentacion porque no entendi un choto
[19:52, 14/9/2026] Caro Franco: no son probabilisticos, no penalizan de por si la perdida
[19:53, 14/9/2026] Caro Franco: arboles que mejoran el arbol anterior
[20:05, 14/9/2026] Caro Franco: Focalizar en learning control parameters
[20:07, 14/9/2026] Caro Franco: El mas critico a optimizar es el learninh rate
[20:17, 14/9/2026] Caro Franco: Pos bagging fraction permite ssmpleo cuando ya no me da la potencia
[20:18, 14/9/2026] Caro Franco: Extra trees para cortar features aleatoreamente y agilizar, si entendi
[20:18, 14/9/2026] Caro Franco: Regularizaciones no funcionan tan bien en este caso
[20:18, 14/9/2026] Caro Franco: Pos de tp con categoricos u otros tipos de v
[20:20, 14/9/2026] Caro Franco: Se pueden usar varios modelos en lugar de arboles
[20:20, 14/9/2026] Caro Franco: Optimiza mas rapido que un rf
[20:21, 14/9/2026] Caro Franco: Le puedo pasar funcion para que optimice ej ganancia en lugar de entropia
[20:21, 14/9/2026] Caro Franco: Le puedo pasar directamente el crossval
[20:24, 14/9/2026] Caro Franco: Siempre recordar que mi top de optimizacion puede no ser mejor en el futuro. Sacar el top modelos optimizados para probar. Siempre
[20:25, 14/9/2026] Caro Franco: Nos reducen el scope pero no nos dan la verdad absoluta. Tatoo de incertidumbre
[20:27, 14/9/2026] Caro Franco: Ademas ese top puedo probarlo con varias semillas porque pude haber tenido suerte
[20:29, 14/9/2026] Caro Franco: Recordar mirar boxplots
[20:31, 14/9/2026] Caro Franco: A dos modelos parecidos les puedo dar mas semillas
[20:36, 14/9/2026] Caro Franco: Ver distrib de probas x clases
[20:37, 14/9/2026] Caro Franco: Hay que buscar baja+3???
[20:43, 14/9/2026] Caro Franco: Revisar lo de ks porque no entendi
[20:43, 14/9/2026] Caro Franco: A mayir ks, mayor capacidad de discriminacion
[20:47, 14/9/2026] Caro Franco: Ver como esta hecho el grafico de ganancia acumulada segun cantidad de clientes estimulados

4.Cómo sincronizar en el día a día:Cuando agregues o edites scripts:Vuelve a la pestaña de Source Control (Ctrl + Shift + G).Escribe una breve nota en el cuadro de texto superior (ej. "agregado script de limpieza").Haz clic en Commit.Haz clic en Sync Changes (o Sync) para enviar los cambios a GitHub.

## Preparación de entorno

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

from scipy.stats import gaussian_kde
from scipy.stats import ks_2samp

from sklearn.model_selection import StratifiedShuffleSplit

import lightgbm as lgb

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour
from optuna.trial import TrialState

from datetime import datetime

from itertools import cycle

from pathlib import Path


c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from pathlib import Path

Importamos data con FE ya hecho

In [2]:
base_path = "/Users/carof/Downloads/DENICOLAY/"
dataset_path = base_path + 'data/' 
modelos_path = base_path + 'modelos/'
db_path = base_path + 'bases_optimizacion/'
bitacora_path = base_path + 'bitacora/'
dataset_file = 'competencia_01_fev2.parquet' #'competencia_01_fe.csv'

data = pd.read_parquet(dataset_path + dataset_file)
#data = pd.read_csv(dataset_path + dataset_file)

In [3]:
# agregue sus semillas
semilla = 42
np.random.seed(semilla)
semillas = np.random.choice(1000000, size=50, replace=False).tolist()

## Separación test-train / X e y. Varias versiones

In [4]:
data['clase_binaria12'] = np.where(data['clase_ternaria'].isin(['BAJA+1', 'BAJA+2']), 1, 0)  # continua vs resto
data['clase_binaria1'] = np.where(data['clase_ternaria'].isin(['BAJA+1']), 1, 0)  # baja+1 vs resto
data['clase_binaria2'] = np.where(data['clase_ternaria'].isin(['BAJA+2']), 1, 0)  # baja+2 vs resto
# otras posibles..

In [5]:
mes_train = [202103, 202104, 202105]
mes_test = 202106

train_data_abril = data[data['foto_mes'] == 202104]
train_data_marzo_abril = data[data['foto_mes'].isin([202103, 202104])]
train_data_all = data[data['foto_mes'].isin(mes_train)]
test_data = data[data['foto_mes'] == mes_test]

In [6]:
cols_target = ['clase_ternaria', 'clase_binaria12', 'clase_binaria1', 'clase_binaria2']
cols_inutiles = ['foto_mes']
cols_drift = ["cprestamos_personales", "mprestamos_personales"]
cols_a_drop = cols_target + cols_inutiles + cols_drift

# solo abril
X_train_abril = train_data_abril.drop(cols_a_drop, axis=1)
y_train_abril_binaria12 = train_data_abril['clase_binaria12']
y_train_abril_binaria1 = train_data_abril['clase_binaria1']
y_train_abril_binaria2 = train_data_abril['clase_binaria2']

# marzo-abril
X_train_marzo_abril = train_data_marzo_abril.drop(cols_a_drop, axis=1)
y_train_marzo_abril_binaria12 = train_data_marzo_abril['clase_binaria12']
y_train_marzo_abril_binaria1 = train_data_marzo_abril['clase_binaria1']
y_train_marzo_abril_binaria2 = train_data_marzo_abril['clase_binaria2']

# marzo-abril-mayo # all
X_train_all = train_data_all.drop(cols_a_drop, axis=1)
y_train_all_binaria12 = train_data_all['clase_binaria12']
y_train_all_binaria1 = train_data_all['clase_binaria1']
y_train_all_binaria2 = train_data_all['clase_binaria2']

# Test (junio)
X_test = test_data.drop(cols_a_drop, axis=1)
y_test_binaria12 = test_data['clase_binaria12']
y_test_binaria1 = test_data['clase_binaria1']
y_test_binaria2 = test_data['clase_binaria2']
y_test_true = np.where(test_data['clase_ternaria'] == "BAJA+2", 1, 0) # deberia ser idem de arriba # dejo igual para legibilidad
y_test_class = test_data['clase_ternaria']

In [7]:
# Train Datasets - Binaria 12
train_data_lgb_abril_b12 = lgb.Dataset(X_train_abril, label=y_train_abril_binaria12)
train_data_lgb_marzo_abril_b12 = lgb.Dataset(X_train_marzo_abril, label=y_train_marzo_abril_binaria12)
train_data_lgb_all_b12 = lgb.Dataset(X_train_all, label=y_train_all_binaria12)

# Train Datasets - Binaria 1
train_data_lgb_abril_b1 = lgb.Dataset(X_train_abril, label=y_train_abril_binaria1)
train_data_lgb_marzo_abril_b1 = lgb.Dataset(X_train_marzo_abril, label=y_train_marzo_abril_binaria1)
train_data_lgb_all_b1 = lgb.Dataset(X_train_all, label=y_train_all_binaria1)

# Train Datasets - Binaria 2
train_data_lgb_abril_b2 = lgb.Dataset(X_train_abril, label=y_train_abril_binaria2)
train_data_lgb_marzo_abril_b2 = lgb.Dataset(X_train_marzo_abril, label=y_train_marzo_abril_binaria2)
train_data_lgb_all_b2 = lgb.Dataset(X_train_all, label=y_train_all_binaria2)

# Test Datasets (Junio)
test_data_lgb_b12 = lgb.Dataset(X_test, label=y_test_binaria12, reference=train_data_lgb_all_b12)
test_data_lgb_b1 = lgb.Dataset(X_test, label=y_test_binaria1, reference=train_data_lgb_all_b1)
test_data_lgb_b2 = lgb.Dataset(X_test, label=y_test_binaria2, reference=train_data_lgb_all_b2)

In [26]:
ganancia_nominal = 2_200_000
efectividad = 0.5
costo_estimulo = 27_500 # Lo que cuesta enviar la campaña/estímulo a un cliente. Si se lo envías a alguien que no se va a quedar (y=0), ese costo se pierde (un Falso Positivo).
beneficio_bruto_acierto = ganancia_nominal * efectividad
ganancia_acierto = ganancia_nominal * efectividad - costo_estimulo
#ganancia_acierto = 1_072_500# El beneficio neto que se obtiene si estimulas a un cliente positivo (y=1) y este se queda (un Verdadero Positivo)

# umbral definido en train para cuando trabajamos con clases combinadas:
prop_baja2_abril = y_train_abril_binaria2.sum() / y_train_abril_binaria12.sum() # binaria2 = 1 si baja+2, else 0. binaria12 = 1 si baja+2 | baja+1, else 0. 
umbral12_abril = costo_estimulo / (prop_baja2_abril * beneficio_bruto_acierto)
print(umbral12_abril)

prop_baja2_marzo_abril = y_train_marzo_abril_binaria2.sum() / y_train_marzo_abril_binaria12.sum() # binaria2 = 1 si baja+2, else 0. binaria12 = 1 si baja+2 | baja+1, else 0. 
umbral12_marzo_abril = costo_estimulo / (prop_baja2_marzo_abril * beneficio_bruto_acierto)
print(umbral12_marzo_abril)

prop_baja2_all = y_train_all_binaria2.sum() / y_train_all_binaria12.sum() # binaria2 = 1 si baja+2, else 0. binaria12 = 1 si baja+2 | baja+1, else 0. 
umbral12_all = costo_estimulo / (prop_baja2_all * beneficio_bruto_acierto)
print(umbral12_all)

# umbral definido para cuando trabajamos solo con baja+2
umbral2 = costo_estimulo / beneficio_bruto_acierto
print(umbral2)

prop_val = 0.3

0.046005267778753293
0.048523106241067177
0.05122094981475245
0.025


## Función de ganancia

OJO: AL FINAL DEFINIMOS FUNCION ADENTRO DE FUNCION DE ABAJO

In [11]:
def ganancia_prob_lgm_binaria2(y_pred, data, umbral = umbral2, prop_validation=prop_val):
  "Funcion para calcular la ganancia monetaria de nuestras predicciones, específicamente adaptada a lightgmb"
  "y pred = vector de probas predichas por el modelo."
  "data = dataset interno de evaluación del modelo"
  "umbral = umbral de probabilidad predicha a partir del cual envio el estímulo (i.e: considero a un cliente una baja potencial)."
  "prop_validación = la proporción que representa el dataset de evaluación respecto del total, para poder escalar bien el resultado. "
  # Paso 1: Extrae el vector de valores reales del dataset (1 o 0) para contrastarlos con las predicciones
  y_true = data.get_label() 
  # Paso 2: Cálculo de cuánto dinero aporta o resta cada cliente si decidieras enviarle el estímulo:
  # Si se los envío a los true 1, gano la ganancia_acierto; si se los envío a los true = 0, pierdo el costo_estímulo
  # np.where funciona como : condicion, true, false
  ganancia = np.where(y_true == 1, ganancia_acierto, 0) - np.where(y_true == 0, costo_estimulo, 0) 
  # Tercer paso: 
  # Suma las ganancias y pérdidas de todos los clientes que superaron el corte de 0.025 en el conjunto de validación 
  # Es decir, a los que efectivamente envié el estimulo, se fija si eran true o false positive y toma su ganancia o costo adjunto, para luego sumarlos
  # Aparte, como la validación solo tiene prop_validation de los datos, dividir por esa proporción proyecta cuál sería la ganancia total equivalente si se aplicara sobre el 100% del universo de datos
  # Finalmente, formatea en: 'nombre', valor, True --> donde True equivale a "más alto, mejor"
  return 'gan_prob', ganancia[y_pred >= umbral].sum() / prop_validation, True

In [20]:
# y_true: vector con 1 SOLO para baja+2, y 0 para el resto (incluyendo baja+1)
#beneficio_bruto_acierto = ganancia_nominal * efectividad # 1_100_000

def ganancia_prob_lgm_binaria12_ytrue(y_pred, data, y_true, umbral=umbral12_all, prop_validation=prop_val):
    "y_true: vector con 1 SOLO para baja+2, y 0 para el resto (incluyendo baja+1)"
    "beneficio_bruto_acierto  = ganancia_nominal * efectividad # 1_100_000"
    " TENGO QUE PODER PASAR UN UMBRAL + LA Y TRUE"
    envios = (y_pred >= umbral)
    # 1. Ingresos brutos: solo aportan los baja+2 efectivamente estimulados
    ingresos = (envios & (y_true == 1)).sum() * beneficio_bruto_acierto
    # 2. Costos: se paga por CADA cliente al que se le envió el estímulo
    costos = envios.sum() * costo_estimulo
    ganancia_neta = (ingresos - costos) / prop_validation
    return 'gan_prob', ganancia_neta, True

In [21]:
# cuando no puedo pasar vector de y_true

def ganancia_prob_lgm_binaria12_prop(y_pred, data, umbral=umbral12_all, prop_baja2 = prop_baja2_all, prop_validation=prop_val):
    " TENGO QUE PODER PASAR UN UMBRAL + LA PROP DE BAJA2 ENTRE LOS BAJAS "
    y_true = data.get_label() # 1 si (baja+1 | baja+2), 0 si continúa

    ganancia_acierto_esperada = ganancia_nominal * efectividad * prop_baja2_all - costo_estimulo 
    # Si estimulas a un y_true == 1, ganas en promedio ganancia_acierto_esperada
    # Si estimulas a un y_true == 0, pierdes costo_estimulo
    ganancia_por_cliente = np.where(y_true == 1, ganancia_acierto_esperada, -costo_estimulo)
    
    # Solo sumamos a los que superan el umbral
    ganancia_total = ganancia_por_cliente[y_pred >= umbral].sum() / prop_validation
    
    return 'gan_prob', ganancia_total, True

In [14]:
def gan_prob_general(y_pred, y_true, umbral = umbral2, prop = 1):
  "Funcion para calcular la ganancia monetaria de nuestras predicciones, pensada para chequear el poder de nuestros modelos a futuro"
  ganancia = np.where(y_true == 1, ganancia_acierto, 0) - np.where(y_true == 0, costo_estimulo, 0)
  return ganancia[y_pred >= umbral].sum() / prop

## Optimización OPTUNA

[HIPERPAMS_LIGHTGBM](https://lightgbm.readthedocs.io/en/latest/Parameters.html)

### Definiciones generales

In [25]:
storage_name = f"sqlite:///{db_path}competencia_1.db"

In [16]:
# Obtener la fecha actual con formato MMDD
fecha_str = datetime.now().strftime("%m%d")

In [23]:
FIXED_PARAMS = {
        'objective': 'binary',
        'metric': 'custom',
        'boosting_type': 'gbdt', # el default
        'first_metric_only': True, # default = false, type = bool.LightGBM allows you to provide multiple evaluation metrics. Set this to true, if you want to use only the first metric for early stopping
        'boost_from_average': True, # el default. adjusts initial score to the mean of labels for faster convergence
        'feature_pre_filter': False, # necesario para optimizar min_data_in_leaf set this to true (the default) to tell LightGBM to ignore the features that are unsplittable based on min_data_in_leaf as dataset object is initialized only once and cannot be changed after that, you may need to set this to false when searching parameters with min_data_in_leaf, otherwise features are filtered by min_data_in_leaf firstly if you don’t reconstruct dataset object Note: setting this to false may slow down the training
        'max_bin': 31, # suena a que se podría optimizar. max_bin 🔗︎, default = 255, type = int, aliases: max_bins, constraints: max_bin > 1. max number of bins that feature values will be bucketed in. small number of bins may reduce training accuracy but may increase general power (deal with over-fitting) LightGBM will auto compress memory according to max_bin. For example, LightGBM will use uint8_t for feature value if max_bin=255
        'seed': semillas[0],
        'verbose': -1
}

In [24]:
trials_objetivo = 20

### LOOP por distintos sets de entrenamiento

In [27]:
def create_objective(X, y_train, y_eval_real, precomputed_folds, umbral, prop_val=prop_val):

    y_train = np.asarray(y_train)
    y_eval_real = np.asarray(y_eval_real)

    def objetivo(trial):

        num_leaves = trial.suggest_int('num_leaves', 8, 150)
        learning_rate = trial.suggest_float('learning_rate', 0.005, 0.3) # mas bajo, más iteraciones necesita
        min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 1, 2000)
        feature_fraction = trial.suggest_float('feature_fraction', 0.1, 1.0)
        bagging_fraction = trial.suggest_float('bagging_fraction', 0.1, 1.0)
        bagging_freq = trial.suggest_int('bagging_freq', 0, 1)

        params = {
            **FIXED_PARAMS,
            'num_leaves': num_leaves,
            'learning_rate': learning_rate,
            'min_data_in_leaf': min_data_in_leaf,
            'feature_fraction': feature_fraction,
            'bagging_fraction': bagging_fraction,
            'bagging_freq': bagging_freq,
            'metric': 'auc',
        }

        stopping_rounds = int(50 + 5 / learning_rate)

        ganancias_folds = []
        best_iters = []

        for train_idx, val_idx in precomputed_folds:

            X_tr = X.iloc[train_idx]
            X_val = X.iloc[val_idx]

            y_tr = y_train[train_idx]
            y_val = y_train[val_idx]

            y_val_negocio = y_eval_real[val_idx]

            model = lgb.LGBMClassifier(
                n_estimators=5000, # se puede seguir subiendo
                **params
            )

            # FIT con X e Y = baja+1 + baja+2 (si paso y_binaria12)

            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[
                    lgb.early_stopping(
                        stopping_rounds,
                        verbose=False
                    )
                ]
            )

            # Proba predicha x fold

            y_pred = model.predict_proba(X_val, num_iteration=model.best_iteration_)[:, 1]

            # Cálculo de ganancia:

            envios = y_pred >= umbral
            aciertos = envios & (y_val_negocio == 1) # Y TRUE = BAJA+2 SOLAMENTE
            ingresos = aciertos.sum() * beneficio_bruto_acierto
            costos = envios.sum() * costo_estimulo
            ganancia = (ingresos - costos) / prop_val

            ganancias_folds.append(ganancia)
            best_iters.append(model.best_iteration_)

        ganancia_media = np.mean(ganancias_folds)

        trial.set_user_attr(
            "best_iter", int(np.mean(best_iters)))

        return ganancia_media

    return objetivo

In [ ]:
# VIEJO + APUNTE CON ANOTACIONES
#def create_objective(dtrain, precomputed_folds, y_eval_real, umbral, prop_val=prop_val):
#    """
#        dtrain: Dataset de LightGBM (puede estar entrenado con b12, b1 o b2).
##        precomputed_folds: Lista fija de (train_idx, val_idx) del StratifiedShuffleSplit.
#        y_eval_real: Serie o array con la clase real de negocio (ej. y_train_*_binaria2).
#        umbral = float, umbral de p para enviar estimulo
#        prop_val, FIXED_PARAMS y otras cuestiones están definidas más atrás.
#        """
# #   y_eval_arr = np.array(y_eval_real)
#        
#    # Precalculamos las etiquetas reales solo para las observaciones de cada fold de validación
#    val_labels_by_fold = [y_eval_arr[val_idx] for _, val_idx in precomputed_folds]
#    
#    def objective(trial):
#        # Generador cíclico: entrega las etiquetas reales del fold 0, 1, ..., k-1 en cada ronda
#        fold_iterator = cycle(val_labels_by_fold) 
#        y_true =  # QUE HAGO ACA?? 
#
#        def ganancia_prob_lgm_binaria12_ytrue(y_pred, data, y_true, umbral=umbral, prop_validation=prop_val):
#           "y_true: vector con 1 SOLO para baja+2, y 0 para el resto (incluyendo baja+1)"
#           "beneficio_bruto_acierto  = ganancia_nominal * efectividad # 1_100_000"
#          " TENGO QUE PODER PASAR UN UMBRAL + LA Y TRUE"
#           envios = (y_pred >= umbral)
#           # 1. Ingresos brutos: solo aportan los baja+2 efectivamente estimulados
#           ingresos = (envios & (y_true == 1)).sum() * beneficio_bruto_acierto
#           # 2. Costos: se paga por CADA cliente al que se le envió el estímulo
#           costos = envios.sum() * costo_estimulo
#           ganancia_neta = (ingresos - costos) / prop_validation
#           return 'gan_prob', ganancia_neta, True
#
#        num_leaves = trial.suggest_int('num_leaves', 8, 150)
#        learning_rate = trial.suggest_float('learning_rate', 0.005, 0.3) # mas bajo, más iteraciones necesita
#        min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 1, 2000)
#        feature_fraction = trial.suggest_float('feature_fraction', 0.1, 1.0)
#        bagging_fraction = trial.suggest_float('bagging_fraction', 0.1, 1.0)
#        bagging_freq = trial.suggest_int('bagging_freq', 0, 1)

#        params = {
#            **FIXED_PARAMS,
#            'num_leaves': num_leaves, # num_leaves 🔗︎, default = 31, type = int, constraints: 1 < num_leaves <= 131072. max number of leaves in one tree
#            'learning_rate': learning_rate, # learning_rate 🔗︎, default = 0.1, type = double, constraints: learning_rate > 0.0. shrinkage rate
#            'min_data_in_leaf': min_data_in_leaf, # min_data_in_leaf 🔗︎, default = 20, type = int, aliases: min_data_per_leaf, min_data, min_child_samples, min_samples_leaf, constraints: min_data_in_leaf >= 0 minimal number of data in one leaf. Can be used to deal with over-fitting Note: this is an approximation based on the Hessian, so occasionally you may observe splits which produce leaf nodes that have less than this many observations
#            'feature_fraction': feature_fraction, # feature_fraction 🔗︎, default = 1.0, type = double, constraints: 0.0 < feature_fraction <= 1.0. LightGBM will randomly select a subset of features on each iteration (tree) if feature_fraction is smaller than 1.0. For example, if you set it to 0.8, LightGBM will select 80% of features before training each tree can be used to speed up training can be used to deal with over-fitting
#            'bagging_fraction': bagging_fraction, # bagging_fraction 🔗︎, default = 1.0, type = double, aliases: sub_row, subsample, bagging, constraints: 0.0 < bagging_fraction <= 1.0 like feature_fraction, but this will randomly select part of data without resampling can be used to speed up training can be used to deal with over-fitting Note: to enable bagging, bagging_freq should be set to a non zero value as well
#            # OJOTA: , deberiamos optimizar este parametro tb para que el de arriba tenga sentido... bagging_freq 🔗︎, default = 0, type = int, aliases: subsample_freq frequency for bagging 0 means disable bagging; k means perform bagging at every k iteration. Every k-th iteration, LightGBM will randomly select bagging_fraction * 100% of the data to use for the next k iterations
#            'bagging_freq' : bagging_freq
 #       }

#        # Early stopping adaptativo a la tasa de aprendizaje
#        stopping_rounds = int(50 + 5 / learning_rate)

 #       cv_results = lgb.cv(
 #           params,
 #           dtrain,
 #           num_boost_round=5000, # modificar, subir y subir.. 
 #           # num_iterations 🔗︎, default = 100, type = int, aliases: num_iteration, n_iter, num_tree, num_trees, num_round, num_rounds, nrounds, num_boost_round, n_estimators, max_iter, constraints: num_iterations >= 0
 #           #cnumber of boosting iterations
 #           # Note: internally, LightGBM constructs num_class * num_iterations trees for multi-class classification problems.
 #          #folds=sss_opt.split(X_train, y_train_binaria), # usamos StratifiedShuffleSplit en vez del k-fold interno
 #           folds=precomputed_folds,
 #           callbacks=[lgb.early_stopping(stopping_rounds=stopping_rounds, verbose=False)],
 #           feval=ganancia_prob_lgm_binaria12_ytrue,         
 #           seed=semillas[0]
 #       )
        # Nombre de la métrica retornada por feval
 #       # Ajustar si en tu feval el nombre difiere (ej. 'valid gan_prob-mean') # NO ENTIENDO
 #       metric_key = [k for k in cv_results.keys() if 'gan_prob' in k and 'mean' in k][0]
 #       ganancias = cv_results[metric_key]

 #       max_gan = max(ganancias)
 #       best_iter = ganancias.index(max_gan) + 1

#        # Guardamos cual es la mejor iteración del modelo
 #       trial.set_user_attr("best_iter", best_iter)

  #      return max_gan
   # return objective


Cambio en la función para poder pasar y_true:
1) lgb.LGBMClassifier vs lgb.train Ambos entrenan LightGBM, pero son dos interfaces diferentes. lgb.train Es la API "nativa": model = lgb.train( params, train_data_lgb, num_boost_round=best_iter ) y_pred = model.predict(X_test) Trabaja directamente con: lgb.Dataset(...) lgb.LGBMClassifier Es la API estilo scikit-learn: model = lgb.LGBMClassifier( n_estimators=best_iter, **params ) model.fit(X_train, y_train) y_pred = model.predict_proba(X_test)[:, 1] Trabaja directamente con X e y. Para tu caso Como la optimización nueva la hicimos con LGBMClassifier.fit(), yo seguiría usando LGBMClassifier también para entrenar los modelos finales. No porque sea "mejor", sino porque es más consistente con lo que acabás de optimizar. Importante: los modelos no deberían cambiar de API entre optimización y evaluación final sin motivo. Queremos que el modelo que evaluás sea equivalente al que Optuna estaba evaluando.

### Entrenamiento con baja+1 y baja+2 juntos

In [28]:
datasets_opt_b12 = {
    'abril_b12': {
        'lgb_data': train_data_lgb_abril_b12,
        'X': X_train_abril,
        'y': y_train_abril_binaria12,
        'y_eval' : y_train_abril_binaria2,
        'umbral' : umbral12_abril,
    },
    'marzo_abril_b12': {
        'lgb_data': train_data_lgb_marzo_abril_b12,
        'X': X_train_marzo_abril,
        'y': y_train_marzo_abril_binaria12,
        'y_eval' : y_train_marzo_abril_binaria2,
        'umbral' : umbral12_marzo_abril,
    },
    'all_b12': {
        'lgb_data': train_data_lgb_all_b12,
        'X': X_train_all,
        'y': y_train_all_binaria12,
        'y_eval' : y_train_all_binaria2,
        'umbral' : umbral12_all,
    }
}

In [29]:
# Loop de optimización
# fecha_str, storage_name y prop_val definidos arriba
estudios = {}
splits = 5

for name, cfg in datasets_opt_b12.items():
    print(f"\n{'='*40}")
    print(f"Iniciando optimización: {name}")
    print(f"{'='*40}")
    study_name = f"optimizacion_{name}_{fecha_str}"

    # Generamos y fijamos los folds en memoria para este dataset
    sss_opt = StratifiedShuffleSplit(n_splits=splits, test_size=prop_val, random_state=semillas[0])
    precomputed_folds = list(sss_opt.split(cfg['X'], cfg['y']))

    study = optuna.create_study(
        direction="maximize",
        study_name=study_name,
        storage=storage_name,
        load_if_exists=True,
    )
 
    # Optimizamos pasando el closure con los datos y folds específicos
    # def create_objective(X, y_train, y_eval_real, precomputed_folds, umbral, prop_val=prop_val):
    objective_fn = create_objective(X = cfg['X'], y_train = cfg['y'], y_eval_real =  cfg['y_eval'], precomputed_folds = precomputed_folds, umbral = cfg['umbral'])
    study.optimize(objective_fn, n_trials=trials_objetivo)

    estudios[name] = study
    best_it = study.best_trial.user_attrs.get('best_iter')

    print(f"--> Finalizado {name} | Mejor Ganancia: ${study.best_value:,.2f} | Mejor Iteración: {best_it}")


Iniciando optimización: abril_b12


[I 2026-10-01 12:31:23,298] A new study created in RDB with name: optimizacion_abril_b12_1001
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site

--> Finalizado abril_b12 | Mejor Ganancia: $534,783,333.33 | Mejor Iteración: 158

Iniciando optimización: marzo_abril_b12


[I 2026-10-01 12:46:54,047] A new study created in RDB with name: optimizacion_marzo_abril_b12_1001
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Li

--> Finalizado marzo_abril_b12 | Mejor Ganancia: $913,330,000.00 | Mejor Iteración: 1622

Iniciando optimización: all_b12


[I 2026-10-01 13:23:03,563] A new study created in RDB with name: optimizacion_all_b12_1001
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-p

KeyboardInterrupt: 

Pending (warning): cambiar 
eval_set=[(X_val, y_val)],  
por 
eval_X=X_val,
eval_y=y_val,

### Para correr estudio único y por primera vez:

In [ ]:
# DEFINIR FUNCION OBJECTIVE

In [ ]:
#study.optimize(objective, n_trials=trials_objetivo)

[I 2026-09-28 18:27:49,513] Trial 0 finished with value: 4010930000.0 and parameters: {'num_leaves': 60, 'learning_rate': 0.019151643878698853, 'min_data_in_leaf': 682, 'feature_fraction': 0.600296269830609, 'bagging_fraction': 0.37094299760913263, 'bagging_freq': 0}. Best is trial 0 with value: 4010930000.0.
[I 2026-09-28 18:28:10,691] Trial 1 finished with value: 3206060000.0 and parameters: {'num_leaves': 27, 'learning_rate': 0.2027571737902023, 'min_data_in_leaf': 114, 'feature_fraction': 0.5691399549628874, 'bagging_fraction': 0.17052937027818915, 'bagging_freq': 1}. Best is trial 0 with value: 4010930000.0.
[I 2026-09-28 18:30:01,535] Trial 2 finished with value: 3996996666.6666665 and parameters: {'num_leaves': 53, 'learning_rate': 0.05346959223307288, 'min_data_in_leaf': 1477, 'feature_fraction': 0.5135020996328518, 'bagging_fraction': 0.9107242446393918, 'bagging_freq': 1}. Best is trial 0 with value: 4010930000.0.
[I 2026-09-28 18:33:11,249] Trial 3 finished with value: 39424

### Para correr estudio ya guardado

In [35]:
# Cargar el estudio existente
saved_study_name = "optimizacion_all_b12_1001" # AJUSTAR SEGUN CORRESPONDA
cfg = datasets_opt_b12['all_b12'] # AJUSTAR SEGUN CORRESPONDA
study = optuna.load_study(study_name=saved_study_name, storage=storage_name)

In [36]:
# Continuo entrenando
trials_pendientes = max(trials_objetivo - len(study.trials), 0)

if trials_pendientes > 0:
    #study.optimize(objective, n_trials=trials_pendientes)
    sss_opt = StratifiedShuffleSplit(n_splits=splits, test_size=prop_val, random_state=semillas[0])
    precomputed_folds = list(sss_opt.split(cfg['X'], cfg['y']))
    objective_fn = create_objective(X = cfg['X'], y_train = cfg['y'], y_eval_real =  cfg['y_eval'], precomputed_folds = precomputed_folds, umbral = cfg['umbral'])
    study.optimize(objective_fn, n_trials=trials_pendientes)

c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

### Resultados Optuna

Visualizo resultados de optimización

In [45]:
optuna.visualization.plot_optimization_history(study)

In [38]:
plot_param_importances(study)

In [39]:
plot_slice(study)

In [40]:
plot_contour(study)

## Extraigo top modelos, entreno y guardo

In [46]:
top_n = 5

In [ ]:
estudios = {
    'abril_b12': {
        'study': optuna.load_study(study_name="optimizacion_abril_b12_1001", storage=storage_name),
        'lgb_data': train_data_lgb_abril_b12,
        'X': X_train_abril,
        'y': y_train_abril_binaria12,
        'y_eval' : y_train_abril_binaria2,
        'umbral' : umbral12_abril,
    },
    'marzo_abril_b12': {
        'study': optuna.load_study(study_name="optimizacion_marzo_abril_b12_1001", storage=storage_name),
        'lgb_data': train_data_lgb_marzo_abril_b12,
        'X': X_train_marzo_abril,
        'y': y_train_marzo_abril_binaria12,
        'y_eval' : y_train_marzo_abril_binaria2,
        'umbral' : umbral12_marzo_abril,
    },
    'all_b12': {
        'study': optuna.load_study(study_name="optimizacion_all_b12_1001", storage=storage_name),
        'lgb_data': train_data_lgb_all_b12,
        'X': X_train_all,
        'y': y_train_all_binaria12,
        'y_eval' : y_train_all_binaria2,
        'umbral' : umbral12_all,
    }
}

In [ ]:
resultados = []

for study_name, study in estudios.items():

    completed_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

    top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:top_n]

    cfg = datasets_opt_b12[study_name]

    X_train = cfg['X']
    y_train = cfg['y']

    for i, trial in enumerate(top_trials, start=1):

        best_iter = trial.user_attrs.get("best_iter", 100)

        params = {**FIXED_PARAMS, **trial.params}

        model = lgb.LGBMClassifier(
            n_estimators=best_iter,
            **params
        )

        model.fit(X_train, y_train)

        y_pred_test = model.predict_proba(X_test)[:, 1]
        ganancia_test = gan_prob_general(y_pred_test, y_test_true)

        top_15_features = (
            pd.Series(
                model.feature_importances_,
                index=X_train.columns
            )
            .nlargest(15)
            .index
            .tolist()
        )

        resultados.append({
            "study": study_name,
            "fecha": fecha_str,
            "orden": i,
            "trial": trial.number,
            "valor_cv": trial.value,
            "best_iter": best_iter,
            "ganancia_test": ganancia_test,
            "envios_test": int((y_pred_test >= cfg['umbral']).sum()),
            "umbral": cfg['umbral'],
            "baja2_capturados_test": int(((y_pred_test >= cfg['umbral']) & (y_test_true == 1)).sum()),
            "precision_baja2_test": (((y_pred_test >= cfg['umbral']) & (y_test_true == 1)).sum() / max((y_pred_test >= cfg['umbral']).sum(), 1)),
            "top_features": top_15_features,
            **trial.params
        })

df_top_models = pd.DataFrame(resultados)

In [ ]:
nombre_archivo = f"{bitacora_path}BitacoraTop{top_n}Modelos_{fecha_str}.csv"
df_top_models.to_csv(nombre_archivo)

In [ ]:
# Para cargar todas las bitacoras disponibles

archivos = Path(bitacora_path).glob("*.csv")
df_bitacora = pd.concat((pd.read_csv(f).assign(archivo=f.name) for f in archivos), ignore_index=True)

df_bitacora

In [ ]:
# APUNTES / VIEJO
# Caso de 1 unico study
#  Filtro los mejores modelos entre los trials exitosos/completados
# top n definido arriba
#completed_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])
#top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:top_n]

# Entreno en loop
# arriba tengo definido:
#X_train = train_data.drop(['clase_ternaria', 'clase_binaria'], axis=1)
#y_train_binaria = train_data['clase_binaria']
#X_test = test_data.drop(['clase_ternaria', 'clase_binaria'], axis=1)
#y_test_binaria = test_data['clase_binaria']
#y_test_true = np.where(test_data['clase_ternaria'] == "BAJA+2", 1, 0)
#y_test_class = test_data['clase_ternaria']
#train_data_lgb = lgb.Dataset(X_train, label=y_train_binaria)
#test_data_lgb = lgb.Dataset(X_test, label=y_test_binaria)

#resultados = []

#for i, trial in enumerate(top_trials, start=1):
#    best_iter = trial.user_attrs.get("best_iter", 100) 
#
#    params = {**FIXED_PARAMS, **trial.params}

#    model = lgb.train(params, train_data_lgb, num_boost_round=best_iter)
#    filename = f"{modelos_path}modelolgb_{fecha_str}_rank{i}_trial{trial.number}_cv{trial.value:.2f}.txt"
#    model.save_model(filename)

#    y_pred_test = model.predict(X_test)
#    ganancia_test = gan_prob_general(y_pred_test, y_test_true)

    # Top 15 nombres de variables como lista:
#    top_15_features = pd.Series(model.feature_importance(importance_type='gain'), index=X_train.columns).nlargest(15).index.tolist()

#    resultados.append({
#        "orden": i,
#        "trial": trial.number,
#        "valor_cv": trial.value,
#        "best_iter": best_iter,
#        "ganancia_test": ganancia_test,
#        'top_features': top_15_features,
###    })
    
    

#df_top_models = pd.DataFrame(resultados)
#df_top_models

,orden,trial,valor_cv,best_iter,ganancia_test,top_features,num_leaves,learning_rate,min_data_in_leaf,feature_fraction,bagging_fraction,bagging_freq
0,1,0,4.010930e+09,1177,387227500.0,"[ctrx_quarter, ctrx_quarter_min6m, saldo_liqui...",60,0.019152,682,0.600296,0.370943,0
1,2,2,3.996997e+09,366,387145000.0,"[ctrx_quarter, saldo_liquido_total, ctrx_quart...",53,0.053470,1477,0.513502,0.910724,1
2,3,4,3.981670e+09,344,370122500.0,"[ctrx_quarter, saldo_liquido_total, ctrx_quart...",55,0.085993,1520,0.329739,0.286058,0
3,4,9,3.980643e+09,440,400895000.0,"[ctrx_quarter, ctrx_quarter_min6m, saldo_liqui...",110,0.027412,536,0.899391,0.203599,0
4,5,3,3.942437e+09,1375,377905000.0,"[ctrx_quarter, ctrx_quarter_avg3m, mpasivos_ma...",12,0.044197,1843,0.122920,0.513039,0


## Varias semillas x modelo, para extraer distribución

In [54]:
n_semillas = 20

# Obtenemos las claves optimizadas directamente desde el primer trial (o study)
param_keys = list(top_trials[0].params.keys())

rows = []
 
# Iteras sobre los trials directamente: trial.params YA tiene los tipos exactos y automáticos
for trial in top_trials:
    best_iter = trial.user_attrs.get("best_iter", 100)

    for seed in semillas[:n_semillas]:
        params = {
            **FIXED_PARAMS,
            **trial.params,      
            'seed': seed       
        }

        model = lgb.train(params, train_data_lgb, num_boost_round=best_iter)

        y_pred_futura = model.predict(X_test)
        ganancia_futura = gan_prob_general(y_pred_futura, y_test_true)

        rows.append({
            "trial": trial.number,
            "seed": seed,
            "ganancia_mayo": ganancia_futura
        })

df_semillas = pd.DataFrame(rows)

estadisticas = (
    df_semillas.groupby('trial')['ganancia_mayo']
    .agg(['mean', 'std', 'min', 'max', 'median'])
    .sort_values('mean', ascending=False)
)


In [55]:
orden_trials = [str(t) for t in estadisticas.index]

fig = go.Figure()

for trial in estadisticas.index:
    datos_trial = df_semillas[df_semillas['trial'] == trial]
    fig.add_trace(go.Box(
        y=datos_trial['ganancia_mayo'],
        x=[str(trial)] * len(datos_trial),
        name=str(trial),
        boxpoints=False,
        showlegend=False
    ))

primera_semilla = df_semillas[df_semillas['seed'] == semillas[0]]
fig.add_trace(go.Scatter(
    x=[str(t) for t in primera_semilla['trial']],
    y=primera_semilla['ganancia_mayo'],
    mode='markers',
    marker=dict(color='red', size=12, line=dict(color='black', width=1)),
    name=f'semilla {semillas[0]}'
))

fig.update_layout(
    title=f"Ganancia en Mayo por modelo ({n_semillas} semillas)",
    xaxis_title="Trial (modelo)",
    yaxis_title="Ganancia en Mayo",
    xaxis=dict(categoryorder='array', categoryarray=orden_trials),
    template='plotly_white'
)

fig.show()

## Retengo y_pred de modelo/s preferido/s

In [ ]:
# 1. Eliges el número del trial según las estadísticas 
preferred_trial_num = 9 #estadisticas.index[0]  # O ingresas a mano: preferred_trial_num = 14

In [ ]:
# Entreno 
# 2. Recuperas el objeto Trial directamente desde el study
chosen_trial = study.trials[preferred_trial_num]

# 3. Extraes la mejor iteración guardada en user_attrs
model_best_iter = chosen_trial.user_attrs.get("best_iter", 100)

# 4. Fusionas los fijos con TODOS los que Optuna haya optimizado
model_params = {
    **FIXED_PARAMS,
    **chosen_trial.params
}

# 5. Entrenas el modelo final
my_model = lgb.train(
    model_params,
    train_data_lgb,
    num_boost_round=model_best_iter
)

# 6. Predicción de probabilidades para evaluar umbrales
y_pred_my_model = my_model.predict(X_test)

## Evaluo puntos de corte

In [ ]:
import plotly.graph_objects as go

orden = np.argsort(y_pred_my_model)[::-1] # ordeno los valores predichos
print(orden)
ganancia_ordenada = np.where(y_test_class.values == "BAJA+2", ganancia_acierto, -costo_estimulo)[orden]
print(ganancia_ordenada)
ganancia_acumulada = np.cumsum(ganancia_ordenada)

mejor_corte = np.argmax(ganancia_acumulada) + 1
# sumamos "1" para que sea legible en términos de " cantidad de personas a las que conviene enviar "
mejor_ganancia = ganancia_acumulada[mejor_corte - 1] # restamos 1 para recuperar el índice
p_corte = y_pred_my_model[orden][mejor_corte - 1]

limite = 20000 # cortamos la figura para no plotear a todo el n del dataset

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(1, limite + 1)), y=ganancia_acumulada[:limite],
    mode='lines', name='Ganancia acumulada', line=dict(color='blue')
))

fig.add_vline(
    x=mejor_corte, line_dash='dash', line_color='black',
    annotation_text=f'Corte óptimo: {mejor_corte} clientes',
    annotation_position='top'
)

fig.add_trace(go.Scatter(
    x=[mejor_corte], y=[mejor_ganancia],
    mode='markers+text', marker=dict(color='red', size=10),
    text=[f'p = {p_corte:.4f}'], textposition='middle right',
    showlegend=False
))

fig.update_layout(
    title='Ganancia acumulada según cantidad de clientes estimulados',
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    xaxis_range=[0, limite],
    template='plotly_white'
)

fig.show()

print(f"Máxima ganancia: {mejor_ganancia:.2f} en el cliente núm. {mejor_corte}, con probabilidad de corte p = {p_corte:.4f}")

[74914 44501 64750 ... 59986 66140 12329]
[-27500. -27500. -27500. ... -27500. -27500. -27500.]


Máxima ganancia: 405735000.00 en el cliente núm. 12846, con probabilidad de corte p = 0.0232


In [62]:
corte_min = 4000
corte_max = 12000

paso = 500

cortes_tabla = np.arange(corte_min + paso, corte_max + 1, paso) # rango de cortes
print(cortes_tabla)
print(len(ganancia_acumulada))
print(len(y_test_class))
cortes_tabla = cortes_tabla[cortes_tabla <= len(ganancia_acumulada)] # restriccion: no puedo tener más cortes que n?
print(cortes_tabla)

tabla_ganancia = pd.DataFrame({
    'corte': cortes_tabla,
    'ganancia_acumulada': ganancia_acumulada[cortes_tabla - 1] # extraigo ganancia acumulada en el punto de corte
})

tabla_ganancia

[ 4500  5000  5500  6000  6500  7000  7500  8000  8500  9000  9500 10000
 10500 11000 11500 12000]
164114
164114
[ 4500  5000  5500  6000  6500  7000  7500  8000  8500  9000  9500 10000
 10500 11000 11500 12000]


,corte,ganancia_acumulada
0,4500,287650000.0
1,5000,314600000.0
2,5500,322850000.0
3,6000,328900000.0
4,6500,339350000.0
5,7000,345400000.0
6,7500,359150000.0
7,8000,365200000.0
8,8500,369050000.0
9,9000,381700000.0


In [ ]:
n_semillas = 20
# definir lim min y lim max en función del gráfico anterior
lim_min = 5000
lim_max = 20001
cortes = np.arange(lim_min, lim_max, 500)

rows = []

# Preferred trial definido arriba 
# Recuperas el objeto Trial directamente desde el study
chosen_trial = study.trials[preferred_trial_num]

# Extraes la mejor iteración guardada en user_attrs
model_best_iter = chosen_trial.user_attrs.get("best_iter", 100)

# Fusionas los fijos con TODOS los que Optuna haya optimizado
model_params = {
    **FIXED_PARAMS,
    **chosen_trial.params
}

# Convertir vector de clases reales a numpy una sola vez afuera del loop
# (usar y_test_true si solo ganas con BAJA+2)
y_real_eval = np.array(y_test_true)

for seed in semillas[:n_semillas]:
    params_seed = {**model_params, 'seed': seed}  # Copiamos los parámetros del mejor modelo + # agrego semilla a los parametros del modelo

    model_seed = lgb.train(params_seed, train_data_lgb, num_boost_round=model_best_iter) # reentreno
    y_pred_seed = model_seed.predict(X_test)

    orden_seed = np.argsort(y_pred_seed)[::-1]
    ganancia_ordenada_seed = np.where(y_real_eval[orden_seed] == 1, ganancia_acierto, -costo_estimulo)
    ganancia_acumulada_seed = np.cumsum(ganancia_ordenada_seed)

    for corte in cortes:
        rows.append({
            "seed": seed,
            "corte": corte,
            "ganancia": ganancia_acumulada_seed[corte - 1] # veo ganancia acumulada para varios cortes en cada modelo 
        })

df_ganancia_cortes = pd.DataFrame(rows)

estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median'])
)
estadisticas_cortes


,mean,std,min,max,median
corte,,,,,
4000,272195000.0,5.429789e+06,264000000.0,286000000.0,271150000.0
4500,289245000.0,4.229903e+06,282150000.0,300850000.0,289300000.0
5000,306460000.0,5.986775e+06,294800000.0,314600000.0,306350000.0
5500,320100000.0,6.319019e+06,309650000.0,329450000.0,320100000.0
6000,331100000.0,7.477616e+06,315700000.0,344300000.0,330550000.0
6500,343640000.0,5.399503e+06,328350000.0,353650000.0,343750000.0
7000,354255000.0,5.250912e+06,342100000.0,363000000.0,354750000.0
7500,361295000.0,4.604743e+06,354750000.0,371250000.0,360250000.0
8000,367400000.0,4.639533e+06,356400000.0,374000000.0,366300000.0


In [64]:
estadisticas_cortes = ( # visualizo estadistica descriptiva de los cortes
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median',
          lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
)
estadisticas_cortes.columns = ['mean', 'std', 'min', 'max', 'median', 'q1', 'q3']

fig = go.Figure()

fig.add_trace(go.Box(
    x=[str(c) for c in estadisticas_cortes.index],
    q1=estadisticas_cortes['q1'],
    median=estadisticas_cortes['median'],
    q3=estadisticas_cortes['q3'],
    lowerfence=estadisticas_cortes['min'],
    upperfence=estadisticas_cortes['max'],
    mean=estadisticas_cortes['mean'],
    sd=estadisticas_cortes['std'],
    boxmean=True,
))

fig.update_layout(
    title=f'Distribución de ganancia por corte ({n_semillas} semillas)',
    xaxis_title='Cantidad de clientes (corte)',
    yaxis_title='Ganancia acumulada',
    template='plotly_white'
)

fig.show()